# Energy Drink Price Prediction

# Feature Engineering Phase

Objective:
Enhance the cleaned dataset by creating meaningful derived features
that improve machine learning model performance.

In [89]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

print("Libraries loaded successfully")

Libraries loaded successfully


In [90]:
df = pd.read_csv(
    "../data/processed/survey_results_cleaned.csv"
)

print("Shape:", df.shape)

df.head()

Shape: (30000, 17)


,respondent_id,age,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range
0,R00001,30,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150
1,R00002,46,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,Quality,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250
2,R00003,41,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250
3,R00004,33,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200
4,R00005,23,M,Metro,Student,Not Reported,3-4 times,Established,Medium (500 ml),0 to 1,Availability,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100


In [91]:
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

respondent_id                     0
age                               0
gender                            0
zone                              0
occupation                        0
income_levels                     0
consume_frequency(weekly)         0
current_brand                     0
preferable_consumption_size       0
awareness_of_other_brands         0
reasons_for_choosing_brands       0
flavor_preference                 0
purchase_channel                  0
packaging_preference              0
health_concerns                   0
typical_consumption_situations    0
price_range                       0
dtype: int64

Duplicate Rows: 0


## Step 1: Create Age Group Feature

The numerical age column is converted into categorical age groups
to simplify behavioral segmentation and improve model interpretability.

In [92]:
df['age'].describe()

count    30000.000000
mean        33.048167
std         13.438904
min         18.000000
25%         23.000000
50%         31.000000
75%         40.000000
max        604.000000
Name: age, dtype: float64

In [93]:
age_bins = [18, 25, 35, 45, 55, 70, np.inf]

age_labels = [
    '18-25',
    '26-35',
    '36-45',
    '46-55',
    '56-70',
    '70+'
]

df['age_group'] = pd.cut(
    df['age'],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

In [94]:
df['age_group'].value_counts().sort_index()

age_group
18-25    10468
26-35     9093
36-45     5972
46-55     2966
56-70     1492
70+          9
Name: count, dtype: int64

In [95]:
df[['age', 'age_group']].sample(10)

,age,age_group
6614,20,18-25
11257,21,18-25
2676,39,36-45
7530,47,46-55
21012,18,18-25
15196,41,36-45
3158,58,56-70
6431,44,36-45
5733,26,26-35
19867,51,46-55


In [96]:
df.drop(columns='age', inplace=True)

print(df.columns)

Index(['respondent_id', 'gender', 'zone', 'occupation', 'income_levels',
       'consume_frequency(weekly)', 'current_brand',
       'preferable_consumption_size', 'awareness_of_other_brands',
       'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel',
       'packaging_preference', 'health_concerns',
       'typical_consumption_situations', 'price_range', 'age_group'],
      dtype='str')


In [97]:
df.head()

,respondent_id,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group
0,R00001,M,Urban,Working Professional,<10L,3-4 times,Newcomer,Medium (500 ml),0 to 1,Price,Traditional,Online,Simple,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150,26-35
1,R00002,F,Metro,Working Professional,> 35L,5-7 times,Established,Medium (500 ml),2 to 4,Quality,Exotic,Retail Store,Premium,Medium (Moderately health-conscious),Social (eg. Parties),200-250,46-55
2,R00003,F,Rural,Working Professional,> 35L,3-4 times,Newcomer,Medium (500 ml),2 to 4,Availability,Traditional,Retail Store,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",200-250,36-45
3,R00004,F,Urban,Working Professional,16L - 25L,5-7 times,Newcomer,Medium (500 ml),0 to 1,Brand Reputation,Exotic,Online,Eco-Friendly,Low (Not very concerned),"Active (eg. Sports, gym)",150-200,26-35
4,R00005,M,Metro,Student,Not Reported,3-4 times,Established,Medium (500 ml),0 to 1,Availability,Traditional,Online,Premium,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100,18-25


### Observation

- The age column was categorized into six age groups.
- The original numerical age column was removed.
- The new age_group feature improves interpretability and segmentation.

## Step 2: Create Consume Frequency and Awareness Brand Score

A combined behavioral score is created using:

- consume_frequency(weekly)
- awareness_of_other_brands

This score reflects overall engagement with energy drink consumption and market awareness.

In [98]:
consume_freq_map = {
    '0-2 times': 1,
    '3-4 times': 2,
    '5-7 times': 3
}

awareness_map = {
    '0 to 1': 1,
    '2 to 4': 2,
    'above 4': 3
}

In [99]:
df['consume_freq_score'] = (
    df['consume_frequency(weekly)']
    .map(consume_freq_map)
)

df['awareness_score'] = (
    df['awareness_of_other_brands']
    .map(awareness_map)
)

In [100]:
df[[
    'consume_frequency(weekly)',
    'consume_freq_score',
    'awareness_of_other_brands',
    'awareness_score'
]].head(10)

,consume_frequency(weekly),consume_freq_score,awareness_of_other_brands,awareness_score
0,3-4 times,2,0 to 1,1
1,5-7 times,3,2 to 4,2
2,3-4 times,2,2 to 4,2
3,5-7 times,3,0 to 1,1
4,3-4 times,2,0 to 1,1
5,5-7 times,3,2 to 4,2
6,5-7 times,3,2 to 4,2
7,5-7 times,3,above 4,3
8,5-7 times,3,2 to 4,2
9,5-7 times,3,2 to 4,2


In [101]:
df['cf_ab_score'] = (
    df['consume_freq_score'] /
    (
        df['awareness_score'] +
        df['consume_freq_score']
    )
).round(2)

In [102]:
df['cf_ab_score'].describe()

count    30000.000000
mean         0.537318
std          0.141870
min          0.250000
25%          0.500000
50%          0.500000
75%          0.670000
max          0.750000
Name: cf_ab_score, dtype: float64

In [103]:
df['cf_ab_score'].max()

np.float64(0.75)

In [104]:
df.cf_ab_score.value_counts()

cf_ab_score
0.50    10247
0.67     5095
0.75     4043
0.60     3756
0.33     3033
0.40     2261
0.25     1565
Name: count, dtype: int64

In [105]:
sorted(df['cf_ab_score'].unique())

[np.float64(0.25),
 np.float64(0.33),
 np.float64(0.4),
 np.float64(0.5),
 np.float64(0.6),
 np.float64(0.67),
 np.float64(0.75)]

In [106]:
df.drop(
    columns=[
        'consume_freq_score',
        'awareness_score'
    ],
    inplace=True
)

In [107]:
df.columns

Index(['respondent_id', 'gender', 'zone', 'occupation', 'income_levels',
       'consume_frequency(weekly)', 'current_brand',
       'preferable_consumption_size', 'awareness_of_other_brands',
       'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel',
       'packaging_preference', 'health_concerns',
       'typical_consumption_situations', 'price_range', 'age_group',
       'cf_ab_score'],
      dtype='str')

### Observation

- consume_frequency(weekly) and awareness_of_other_brands were converted into numerical scores.
- These scores were combined to create cf_ab_score.
- Temporary helper columns were removed after feature creation.

## Step 3: Create Zone Affluence Score (ZAS)

A combined affluence score is created using:

- geographic zone
- income level

This feature captures the respondent’s socioeconomic profile.

In [108]:
zone_map = {
    'Rural': 1,
    'Semi-Urban': 2,
    'Urban': 3,
    'Metro': 4
}

income_map = {
    '<10L': 1,
    '10L - 15L': 2,
    '16L - 25L': 3,
    '26L - 35L': 4,
    '> 35L': 5,
    'Not Reported': 0
}

In [109]:
df['zone_score'] = df['zone'].map(zone_map)

df['income_score'] = (
    df['income_levels']
    .map(income_map)
)

In [110]:
df[[
    'zone',
    'zone_score',
    'income_levels',
    'income_score'
]].head(10)

,zone,zone_score,income_levels,income_score
0,Urban,3,<10L,1
1,Metro,4,> 35L,5
2,Rural,1,> 35L,5
3,Urban,3,16L - 25L,3
4,Metro,4,Not Reported,0
5,Urban,3,Not Reported,0
6,Urban,3,10L - 15L,2
7,Urban,3,26L - 35L,4
8,Semi-Urban,2,<10L,1
9,Semi-Urban,2,10L - 15L,2


In [111]:
df['zas_score'] = (
    df['zone_score'] *
    df['income_score']
)

In [112]:
df['zas_score'].describe()

count    30000.000000
mean         6.097333
std          5.517953
min          0.000000
25%          0.000000
50%          6.000000
75%          9.000000
max         20.000000
Name: zas_score, dtype: float64

In [113]:
sorted(df['zas_score'].unique())

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(12),
 np.int64(15),
 np.int64(16),
 np.int64(20)]

In [114]:
df.drop(
    columns=[
        'zone_score',
        'income_score'
    ],
    inplace=True
)

In [115]:
df.columns

Index(['respondent_id', 'gender', 'zone', 'occupation', 'income_levels',
       'consume_frequency(weekly)', 'current_brand',
       'preferable_consumption_size', 'awareness_of_other_brands',
       'reasons_for_choosing_brands', 'flavor_preference', 'purchase_channel',
       'packaging_preference', 'health_concerns',
       'typical_consumption_situations', 'price_range', 'age_group',
       'cf_ab_score', 'zas_score'],
      dtype='str')

### Observation

- zone and income_levels were mapped to numerical scores.
- These scores were combined to create zas_score.
- Temporary helper columns were removed after feature creation.

## Step 4: Create Brand Switching Indicator (BSI)

A binary indicator is created to identify respondents
who are potentially more likely to switch brands.

Conditions:
- current_brand is not "Established"
- reasons_for_choosing_brands is either:
  - Price
  - Quality

In [116]:
print(df['current_brand'].value_counts())

print("\n")

print(df['reasons_for_choosing_brands'].value_counts())

current_brand
Established    15467
Newcomer       14533
Name: count, dtype: int64


reasons_for_choosing_brands
Price               14141
Availability         6591
Brand Reputation     4665
Quality              4603
Name: count, dtype: int64


In [117]:
df['bsi'] = np.where(
    (
        df['current_brand'] != 'Established'
    ) &
    (
        df['reasons_for_choosing_brands']
        .isin(['Price', 'Quality'])
    ),
    1,
    0
)

In [118]:
df['bsi'].value_counts()

bsi
0    20822
1     9178
Name: count, dtype: int64

In [119]:
(
    df['bsi']
    .value_counts(normalize=True) * 100
).round(2)

bsi
0    69.41
1    30.59
Name: proportion, dtype: float64

In [120]:
df[df['bsi'] == 1][[
    'current_brand',
    'reasons_for_choosing_brands',
    'bsi'
]].head(10)

,current_brand,reasons_for_choosing_brands,bsi
0,Newcomer,Price,1
6,Newcomer,Price,1
9,Newcomer,Price,1
10,Newcomer,Price,1
18,Newcomer,Quality,1
31,Newcomer,Price,1
34,Newcomer,Price,1
41,Newcomer,Price,1
42,Newcomer,Price,1
44,Newcomer,Price,1


### Observation

- The bsi feature was created as a binary indicator.
- Respondents using non-established brands and prioritizing price/quality were flagged as potential brand switchers.

## Final Cleaning Step: Logical Outlier Detection

Logical inconsistencies between age_group and occupation are identified and removed.

Example:
- Students appearing in older age groups may indicate incorrect survey entries.

In [121]:
pd.crosstab(
    df['age_group'],
    df['occupation']
)

occupation,Entrepreneur,Retired,Student,Working Professional
age_group,,,,
18-25,535,0,7328,2605
26-35,1826,0,697,6570
36-45,1619,0,0,4353
46-55,799,0,0,2167
56-70,221,1130,35,106
70+,1,1,1,6


In [122]:
logical_outliers = df[
    (df['occupation'] == 'Student') &
    (df['age_group'] == '56-70')
]

logical_outliers

,respondent_id,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group,cf_ab_score,zas_score,bsi
182,R00183,F,Urban,Student,Not Reported,5-7 times,Established,Large (1 L),above 4,Price,Exotic,Retail Store,Simple,Medium (Moderately health-conscious),Casual (eg. At home),150-200,56-70,0.50,0,0
3524,R03525,F,Semi-Urban,Student,Not Reported,0-2 times,Established,Medium (500 ml),0 to 1,Price,Traditional,Retail Store,Eco-Friendly,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",50-100,56-70,0.50,0,0
3525,R03526,M,Semi-Urban,Student,Not Reported,0-2 times,Newcomer,Small (250 ml),2 to 4,Price,Exotic,Retail Store,Eco-Friendly,High (Very health-conscious),Casual (eg. At home),100-150,56-70,0.33,0,1
3770,R03771,M,Metro,Student,Not Reported,5-7 times,Newcomer,Medium (500 ml),2 to 4,Availability,Exotic,Online,Simple,High (Very health-conscious),"Active (eg. Sports, gym)",150-200,56-70,0.60,0,0
4031,R04032,M,Urban,Student,Not Reported,3-4 times,Established,Medium (500 ml),0 to 1,Price,Traditional,Retail Store,Simple,High (Very health-conscious),Social (eg. Parties),150-200,56-70,0.67,0,0
6542,R06543,F,Urban,Student,Not Reported,5-7 times,Newcomer,Medium (500 ml),0 to 1,Price,Exotic,Retail Store,Simple,High (Very health-conscious),Social (eg. Parties),150-200,56-70,0.75,0,1
6591,R06592,M,Semi-Urban,Student,Not Reported,5-7 times,Established,Medium (500 ml),0 to 1,Availability,Exotic,Retail Store,Simple,Medium (Moderately health-conscious),Casual (eg. At home),100-150,56-70,0.75,0,0
6645,R06646,F,Semi-Urban,Student,Not Reported,0-2 times,Established,Medium (500 ml),2 to 4,Price,Exotic,Retail Store,Premium,High (Very health-conscious),Casual (eg. At home),100-150,56-70,0.33,0,0
7417,R07418,F,Rural,Student,Not Reported,3-4 times,Established,Small (250 ml),0 to 1,Price,Exotic,Retail Store,Eco-Friendly,Medium (Moderately health-conscious),"Active (eg. Sports, gym)",100-150,56-70,0.67,0,0
7593,R07594,M,Urban,Student,Not Reported,5-7 times,Newcomer,Medium (500 ml),2 to 4,Price,Exotic,Online,Simple,High (Very health-conscious),"Active (eg. Sports, gym)",200-250,56-70,0.60,0,1


In [123]:
print(
    "Logical outliers:",
    len(logical_outliers)
)

Logical outliers: 35


In [124]:
rows_before = len(df)

df = df[
    ~(
        (df['occupation'] == 'Student') &
        (df['age_group'] == '56-70')
    )
]

rows_after = len(df)

print("Rows removed:", rows_before - rows_after)

print("New shape:", df.shape)

Rows removed: 35
New shape: (29965, 20)


In [125]:
df[
    (df['occupation'] == 'Student') &
    (df['age_group'] == '56-70')
]

,respondent_id,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group,cf_ab_score,zas_score,bsi


In [126]:
rows_before = len(df)

df = df[
    ~(
        (df['occupation'] == 'Student') &
        (
            df['age_group']
            .isin(['56-70'])
        )
    )
]

rows_after = len(df)

print("Rows removed:", rows_before - rows_after)

print("New shape:", df.shape)

Rows removed: 0
New shape: (29965, 20)


In [131]:
df[
    (df['occupation'] == 'Student') &
    (
        df['age_group']
        .isin(['56-70'])
    )
]

,respondent_id,gender,zone,occupation,income_levels,consume_frequency(weekly),current_brand,preferable_consumption_size,awareness_of_other_brands,reasons_for_choosing_brands,flavor_preference,purchase_channel,packaging_preference,health_concerns,typical_consumption_situations,price_range,age_group,cf_ab_score,zas_score,bsi


## Final Dataset Validation

Final checks are performed to ensure:

- No missing values
- No duplicate rows
- Feature engineering completed successfully
- Logical inconsistencies removed

In [128]:
print("Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nColumns:")
print(df.columns.tolist())

Shape: (29965, 20)

Missing Values:


respondent_id                     0
gender                            0
zone                              0
occupation                        0
income_levels                     0
consume_frequency(weekly)         0
current_brand                     0
preferable_consumption_size       0
awareness_of_other_brands         0
reasons_for_choosing_brands       0
flavor_preference                 0
purchase_channel                  0
packaging_preference              0
health_concerns                   0
typical_consumption_situations    0
price_range                       0
age_group                         0
cf_ab_score                       0
zas_score                         0
bsi                               0
dtype: int64

Duplicate Rows:
0

Columns:
['respondent_id', 'gender', 'zone', 'occupation', 'income_levels', 'consume_frequency(weekly)', 'current_brand', 'preferable_consumption_size', 'awareness_of_other_brands', 'reasons_for_choosing_brands', 'flavor_preference', 'purchase_cha

In [129]:
df.to_csv(
    "../data/processed/survey_results_feature_engineered.csv",
    index=False
)

print("Feature engineered dataset saved successfully")

Feature engineered dataset saved successfully
